# 09 Deep Learning — Interactive Text Prediction

This notebook allows you to input custom SaaS sales conversation text, along with its expected outcome, and see exactly how the Deep Learning model (BiLSTM + GloVe) processes it and predicts the outcome.

### Steps:
1. **Load Tokenizer**: Loads the custom word-level tokenizer from `results/dl_tokenizer.json`.
2. **Load Model**: Loads the trained GloVe-initialized BiLSTM model checkpoint.
3. **Tokenization (Conversion)**: Enter any custom text string to see it converted into token IDs.
4. **Prediction Evaluation**: Run the tokenized sequence through the model to get a WON/LOST probability score and compare it with the actual outcome.

## 1. Setup and Loading Artifacts

In [5]:
import sys
import json
from pathlib import Path

import torch
import numpy as np

# Add project root to path
PROJECT_ROOT = Path('.').resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.deep_learning import (
    LSTMConfig,
    BiLSTMClassifier,
    SalesTokenizer,
    TokenizerConfig,
    get_device
)

RESULTS_DIR = PROJECT_ROOT / 'results'
MODEL_PATH = RESULTS_DIR / 'model_lstm_glove_best.pt'
TOKENIZER_PATH = RESULTS_DIR / 'dl_tokenizer.json'

device = get_device()
print(f'Using Device: {device}')

# Load Tokenizer
with open(TOKENIZER_PATH, 'r') as f:
    tok_data = json.load(f)
    
config = tok_data['config']
tok_config = TokenizerConfig(
    max_vocab=config['max_vocab'],
    max_length=config['max_length'],
    min_freq=config['min_freq'],
)

tokenizer = SalesTokenizer(tok_config)
tokenizer.word2idx = tok_data['word2idx']
tokenizer.idx2word = {v: k for k, v in tokenizer.word2idx.items()}
tokenizer._fitted = True
vocab_size = len(tokenizer.word2idx)

print(f'Tokenizer loaded with vocabulary size: {vocab_size:,}')

# Load Model
lstm_cfg = LSTMConfig(
    vocab_size=vocab_size,
    embed_dim=100,  # GloVe dimension
    hidden_dim=128,
    num_layers=1,
    dropout=0.5,
    bidirectional=True,
    num_classes=1,
    use_attention=True,
)

model = BiLSTMClassifier(lstm_cfg, pad_idx=0)
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True, map_location=device))
model.to(device)
model.eval()
print('BiLSTM + GloVe model loaded successfully.')

Using Device: mps
Tokenizer loaded with vocabulary size: 28,903
BiLSTM + GloVe model loaded successfully.


## 2. Interactive Prediction

Type or paste your SaaS sales text into the `custom_text` variable below. 
Change the `actual_outcome` variable to match the expected outcome (e.g., `'WON'` or `'LOST'`).

When you run the cell, it will show how the tokenizer strips it down into known tokens before feeding it to the model for a sales outcome score, and then check whether the model predicted it correctly.

In [7]:
# =========== CHANGE THESE VALUES =================
custom_text = """
This software is exactly what we dont need.
"""
actual_outcome = "WON"  # Options: "WON", "LOST", or None
# =================================================

print("=== 1. Text Conversion (Tokenization) ===\n")
token_ids = tokenizer.encode(custom_text)
print(f"Raw Token IDs:\n{token_ids}\n")

# Let's show how these token IDs map back to actual words
words = [tokenizer.idx2word.get(i, '<UNK>') for i in token_ids]
print(f"Tokens mapping:\n{words}\n")

print("=== 2. Model Prediction ===\n")
input_tensor = torch.tensor([token_ids], dtype=torch.long).to(device)

with torch.no_grad():
    logits, attn_weights = model(input_tensor)
    logits = logits.squeeze(-1)
    prob = torch.sigmoid(logits).item()

outcome = "WON" if prob > 0.5 else "LOST"

print(f"Predicted Outcome:           {outcome}")
print(f"Probability Score (0 to 1):  {prob:.4f}")
print(f"Probability of WON:          {prob:.1%}")

if actual_outcome is not None:
    is_correct = (outcome == actual_outcome.upper())
    result_text = "✅ CORRECT" if is_correct else "❌ INCORRECT"
    print(f"\nActual Outcome:              {actual_outcome.upper()}")
    print(f"Prediction Match:            {result_text}")

=== 1. Text Conversion (Tokenization) ===

Raw Token IDs:
[86, 801, 15, 658, 19, 10, 11436, 950]

Tokens mapping:
['this', 'software', 'is', 'exactly', 'what', 'we', 'dont', 'need.']

=== 2. Model Prediction ===

Predicted Outcome:           LOST
Probability Score (0 to 1):  0.0526
Probability of WON:          5.3%

Actual Outcome:              WON
Prediction Match:            ❌ INCORRECT
